# Nemotron RAG Serving Lab -- end-to-end walkthrough

This notebook narrates the whole pipeline and runs the CPU-only pieces inline. The GPU-only pieces (fine-tuning, vLLM/SGLang serving) link out to their own notebooks/scripts, since they need a GPU runtime this walkthrough may not have.

Repo: https://github.com/abdelwb/nemotron-rag-serving-lab -- see the top-level README for the full picture and architecture diagram.

In [ ]:
!git clone https://github.com/abdelwb/nemotron-rag-serving-lab.git
%cd nemotron-rag-serving-lab

## Stage 1 -- Fine-tune (Hugging Face + PyTorch)

Run separately, on a T4 GPU runtime:

1. [`finetune/01_prepare_dataset.ipynb`](../finetune/01_prepare_dataset.ipynb)
2. [`finetune/02_lora_finetune.ipynb`](../finetune/02_lora_finetune.ipynb)

Both need a free Hugging Face account + access token. See [`finetune/README.md`](../finetune/README.md).

## Stage 2 -- Serve + benchmark (vLLM vs. SGLang)

Run on a Linux host with an NVIDIA GPU (a separate Colab GPU session kept alive via a terminal, or a cloud GPU box):

```bash
pip install vllm
HF_LORA_REPO=abdelwb/nemotron-mini-4b-daring-anteater-lora bash serving/vllm/serve_vllm.sh &
python serving/vllm/bench_vllm.py --out serving/results/vllm_bench.csv

pip install "sglang[all]"
HF_LORA_REPO=abdelwb/nemotron-mini-4b-daring-anteater-lora bash serving/sglang/serve_sglang.sh &
python serving/sglang/bench_sglang.py --out serving/results/sglang_bench.csv
```

See [`docs/architecture.md`](../docs/architecture.md) for full flags and expected runtimes.

## Stage 3 -- RAG agent (LangChain) -- CPU-only pieces, runnable right here

The sklearn query router needs no GPU and no running server -- it's a good sanity check that the environment is set up correctly before touching the GPU-only stages.

In [ ]:
!pip install -q scikit-learn joblib
%cd rag_agent

from router_baseline_sklearn import train_and_evaluate, classify

pipeline = train_and_evaluate()

for q in [
    "How do I attach a LoRA adapter when serving with vLLM?",
    "Hi, how are you?",
    "What's the current stock price of NVIDIA?",
]:
    print(f"{q!r} -> {classify(q, pipeline=pipeline)}")

The full agent (`rag_agent/agent.py`) additionally needs a FAISS index (`ingest.py`) and a running vLLM/SGLang server from Stage 2 -- see [`rag_agent/README.md`](../rag_agent/README.md).

## Stage 4 -- Performance modeling (scikit-learn + TensorFlow)

Once `serving/results/vllm_bench.csv` and `sglang_bench.csv` have real rows from Stage 2:

```bash
cd perf_modeling
pip install -r requirements.txt
python train_sklearn_regressor.py
python train_tf_model.py
python compare_models.py
```

The last command prints a side-by-side MAE table -- copy it into the top-level README's Results section.